# **Analysis**

## **Digital Elevation Model Without Download - US Testcase**

In [8]:
import os
import rasterio
import pandas

def save_bounds():
    global bounds_of_files
    existing_files = os.listdir(r"Path/To/Files")
    bounds_data = []
    for file_name in existing_files:
        if file_name.endswith('.tif'):
            with rasterio.open(os.path.join(r"Path/To/Files",file_name)) as dataset:
                bounds = dataset.bounds
            bounds_data.append({
                'File Names': file_name,
                'Left Longitude': bounds.left,
                'Lower Latitude': bounds.bottom,
                'Right Longitude': bounds.right,
                'Upper Latitude': bounds.top
            })
    bounds_of_files = pandas.DataFrame(bounds_data)


def identify_sources(latitudes, longitudes):
    global bounds_of_files
    files = bounds_of_files["File Names"].tolist()
    left_longitude = bounds_of_files["Left Longitude"].tolist()
    lower_latitude = bounds_of_files["Lower Latitude"].tolist()
    right_longitude = bounds_of_files["Right Longitude"].tolist()
    upper_latitude = bounds_of_files["Upper Latitude"].tolist()
    references = [None] * len(latitudes)
    
    for i in range(len(references)):
        for j in range(len(files)):
            if (left_longitude[j] <= longitudes[i] <= right_longitude[j] and 
                lower_latitude[j] <= latitudes[i] <= upper_latitude[j]):
                references[i] = files[j]
                break
        if references[i] is None:
            references[i] = None
    return references

import os
import time
import requests
from osgeo import gdal, osr
import numpy
import pandas
from pyproj import Proj, transform

def elevation(latitude, longitude, file_path):
    file_path = os.path.join(r"Path/To/Files",file_path)
    dataset = gdal.Open(file_path)
    dataset_proj = osr.SpatialReference()
    dataset_proj.ImportFromWkt(dataset.GetProjection())
    wgs84 = osr.SpatialReference()
    wgs84.ImportFromEPSG(4326)
    transform_proj = osr.CoordinateTransformation(wgs84, dataset_proj)
    x, y, _ = transform_proj.TransformPoint(longitude, latitude)
    geo_transform = dataset.GetGeoTransform()
    col = int((x - geo_transform[0]) / geo_transform[1])
    row = int((y - geo_transform[3]) / geo_transform[5])
    band = dataset.GetRasterBand(1)
    return(band.ReadAsArray(col, row, 1, 1)[0][0])

bounds_of_files = pandas.DataFrame()

def main():
    index_start_time = time.time()
    save_bounds()
    index_stop_time = time.time()
    index_time = index_stop_time - index_start_time
    index_time = index_time/10000
    coordinates = numpy.loadtxt('Generated Coordinates (US).txt', delimiter = ',')
    longitudes = [lon for lon, lat in coordinates]
    latitudes = [lat for lon, lat in coordinates]
    references = identify_sources(latitudes, longitudes)
    elevations = []
    t = []
    i = 0
    while i < len(references):
        start_time = time.time()
        if references[i] != None:
            elevations.append(elevation(latitudes[i], longitudes[i], references[i]))
        else:
            elevations.append(numpy.nan)
        stop_time = time.time()
        t.append(index_time + stop_time - start_time)
        i += 1
    data = numpy.column_stack((t, elevations))
    numpy.savetxt("DEM Elevations (US).txt", data, delimiter=',', fmt='%.18e')

if __name__ == "__main__":
    main()

In [9]:
%reset -f

## **Digital Elevation Model Without Download - California Testcase**

In [3]:
import os
import rasterio
import pandas

def save_bounds():
    global bounds_of_files
    existing_files = os.listdir(r"Path/To/Files")
    bounds_data = []
    for file_name in existing_files:
        if file_name.endswith('.tif'):
            with rasterio.open(os.path.join(r"Path/To/Files",file_name)) as dataset:
                bounds = dataset.bounds
            bounds_data.append({
                'File Names': file_name,
                'Left Longitude': bounds.left,
                'Lower Latitude': bounds.bottom,
                'Right Longitude': bounds.right,
                'Upper Latitude': bounds.top
            })
    bounds_of_files = pandas.DataFrame(bounds_data)


def identify_sources(latitudes, longitudes):
    global bounds_of_files
    files = bounds_of_files["File Names"].tolist()
    left_longitude = bounds_of_files["Left Longitude"].tolist()
    lower_latitude = bounds_of_files["Lower Latitude"].tolist()
    right_longitude = bounds_of_files["Right Longitude"].tolist()
    upper_latitude = bounds_of_files["Upper Latitude"].tolist()
    references = [None] * len(latitudes)
    
    for i in range(len(references)):
        for j in range(len(files)):
            if (left_longitude[j] <= longitudes[i] <= right_longitude[j] and 
                lower_latitude[j] <= latitudes[i] <= upper_latitude[j]):
                references[i] = files[j]
                break
        if references[i] is None:
            references[i] = None
    return references

import os
import time
import requests
from osgeo import gdal, osr
import numpy
import pandas
from pyproj import Proj, transform

def elevation(latitude, longitude, file_path):
    file_path = os.path.join(r"Path/To/Files",file_path)
    dataset = gdal.Open(file_path)
    dataset_proj = osr.SpatialReference()
    dataset_proj.ImportFromWkt(dataset.GetProjection())
    wgs84 = osr.SpatialReference()
    wgs84.ImportFromEPSG(4326)
    transform_proj = osr.CoordinateTransformation(wgs84, dataset_proj)
    x, y, _ = transform_proj.TransformPoint(longitude, latitude)
    geo_transform = dataset.GetGeoTransform()
    col = int((x - geo_transform[0]) / geo_transform[1])
    row = int((y - geo_transform[3]) / geo_transform[5])
    band = dataset.GetRasterBand(1)
    return(band.ReadAsArray(col, row, 1, 1)[0][0])

bounds_of_files = pandas.DataFrame()

def main():
    index_start_time = time.time()
    save_bounds()
    index_stop_time = time.time()
    index_time = index_stop_time - index_start_time
    index_time = index_time/10000
    coordinates = numpy.loadtxt('Generated Coordinates (California).txt', delimiter = ',')
    longitudes = [lon for lon, lat in coordinates]
    latitudes = [lat for lon, lat in coordinates]
    references = identify_sources(latitudes, longitudes)
    elevations = []
    t = []
    i = 0
    while i < len(references):
        start_time = time.time()
        if references[i] != None:
            elevations.append(elevation(latitudes[i], longitudes[i], references[i]))
        else:
            elevations.append(numpy.nan)
        stop_time = time.time()
        t.append(index_time + stop_time - start_time)
        i += 1
    data = numpy.column_stack((t, elevations))
    numpy.savetxt("DEM Elevations (California).txt", data, delimiter=',', fmt='%.18e')

if __name__ == "__main__":
    main()

In [4]:
%reset -f

## **Google Elevation API - US Testcase**

In [ ]:
import requests
import numpy
import time
import speedtest

# --------------FUNCTIONS-----------------
def get_elevation(x, y, api_key):
    url = "https://maps.googleapis.com/maps/api/elevation/json"
    params = {"locations": f"{y},{x}", "key": api_key}
    retries = 10
    while retries > 0:
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                results = response.json()
                if "results" in results and len(results["results"]) > 0:
                    return results["results"][0]["elevation"]
                else:
                    return None
            else:
                continue
        except Exception as e:
            continue
        retries -= 1
        time.sleep(0.25)  # Wait for 0.25 second before retrying
    return None
# ---------------- END OF FUNCTIONS -------------

t = []
speed = []

def main():
    api_key = "API Key"  
    coordinates = numpy.loadtxt('Generated Coordinates (US).txt', delimiter=',')
    elevations = []
    st = speedtest.Speedtest()
    
    # Initialize counter for API call timing
    api_calls_made = time.time()
    max_calls_per_sec = 100  # Limit to 100 API calls per second
    
    for lon, lat in coordinates:
        download_speed = st.download() * 10e-6  # in Mega bits per second
        upload_speed = st.upload() * 10e-6     # in Mega bits per second
        speed.append((download_speed, upload_speed))
        start_time = time.time()
        elevations.append(get_elevation(lon, lat, api_key))
        stop_time = time.time()
        t.append(stop_time - start_time)
        if t[-1] < 60/6000:
            time.sleep((60/6000)-t[-1])
    
    data = numpy.column_stack((t, speed, elevations))
    numpy.savetxt("Google Elevations (US).txt", data, delimiter=',')

if __name__ == "__main__":
    main()

In [ ]:
%reset -f

## **Google Elevation API - California Testcase**

In [ ]:
import requests
import numpy
import time
import speedtest

# --------------FUNCTIONS-----------------
def get_elevation(x, y, api_key):
    url = "https://maps.googleapis.com/maps/api/elevation/json"
    params = {"locations": f"{y},{x}", "key": api_key}
    retries = 10
    while retries > 0:
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                results = response.json()
                if "results" in results and len(results["results"]) > 0:
                    return results["results"][0]["elevation"]
                else:
                    return None
            else:
                continue
        except Exception as e:
            continue
        retries -= 1
        time.sleep(0.25)  # Wait for 0.25 second before retrying
    return None
# ---------------- END OF FUNCTIONS -------------

t = []
speed = []

def main():
    api_key = "API Key"  
    coordinates = numpy.loadtxt('Generated Coordinates (California).txt', delimiter=',')
    elevations = []
    st = speedtest.Speedtest()
    
    # Initialize counter for API call timing
    api_calls_made = time.time()
    max_calls_per_sec = 100  # Limit to 100 API calls per second
    
    for lon, lat in coordinates:
        download_speed = st.download() * 10e-6  # in bits per second
        upload_speed = st.upload() * 10e-6     # in bits per second
        speed.append((download_speed, upload_speed))
        start_time = time.time()
        elevations.append(get_elevation(lon, lat, api_key))
        stop_time = time.time()
        t.append(stop_time - start_time)
        if t[-1] < 60/6000:
            time.sleep((60/6000)-t[-1])
    
    data = numpy.column_stack((t, speed, elevations))
    numpy.savetxt("Google Elevations (California).txt", data, delimiter=',')

if __name__ == "__main__":
    main()

In [ ]:
%reset -f

## **Elevation Point Query Service - United States Geological Service - US Testcase**

In [ ]:
import requests
import numpy
import time
import speedtest

# --------------FUNCTIONS-----------------
def get_elevation(x, y):
    url = "https://epqs.nationalmap.gov/v1/json?"
    params = {"x": x, "y": y, "units": "Feet"}
    retries = 10
    while retries > 0:
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                vals = response.json()
                return vals["value"]
            else:
                continue
        except Exception as e:
            continue
        retries -= 1
        time.sleep(0.25)  # Wait for 0.25 second before retrying
    return None
# ---------------- END OF FUNCTIONS -------------

t = []
speed  = []
def main():
    coordinates = numpy.loadtxt('Generated Coordinates (US).txt', delimiter = ',')
    elevations = []
    st = speedtest.Speedtest()
    for lon, lat in coordinates:
        download_speed = st.download()  # in bits per second
        upload_speed = st.upload()      # in bits per second
        speed.append((download_speed, upload_speed))
        start_time = time.time()
        elevations.append(get_elevation(lon,lat))
        stop_time = time.time()
        t.append(stop_time - start_time)
    data = numpy.column_stack((t, speed, elevations))
    numpy.savetxt("EPQS Elevations (US).txt", data, delimiter = ',')

if __name__ == "__main__":
    main()

In [ ]:
%reset -f

## **Elevation Point Query Service - United States Geological Service - California Testcase**

In [ ]:
import requests
import numpy
import time
import speedtest

# --------------FUNCTIONS-----------------
def get_elevation(x, y):
    url = "https://epqs.nationalmap.gov/v1/json?"
    params = {"x": x, "y": y, "units": "Feet"}
    retries = 10
    while retries > 0:
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                vals = response.json()
                return vals["value"]
            else:
                continue
        except Exception as e:
            continue
        retries -= 1
        time.sleep(0.25)  # Wait for 0.25 second before retrying
    return None
# ---------------- END OF FUNCTIONS -------------

t = []
speed  = []
def main():
    coordinates = numpy.loadtxt('Generated Coordinates (California).txt', delimiter = ',')
    elevations = []
    st = speedtest.Speedtest()
    for lon, lat in coordinates:
        download_speed = st.download()  # in bits per second
        upload_speed = st.upload()      # in bits per second
        speed.append((download_speed, upload_speed))
        start_time = time.time()
        elevations.append(get_elevation(lon,lat))
        stop_time = time.time()
        t.append(stop_time - start_time)
    data = numpy.column_stack((t, speed, elevations))
    numpy.savetxt("EPQS Elevations (California).txt", data, delimiter = ',')

if __name__ == "__main__":
    main()

In [ ]:
%reset -f